# face recognition dio

Este notebook faz parte do projeto do desafio da DIO. O objetivo é montar um fluxo simples de detecção de faces com OpenCV e simular a etapa de reconhecimento/classificação com labels como `pessoa_1`, `pessoa_2` e `pessoa_3`.

O projeto não treina um modelo do zero. Ele usa um detector pré-treinado para localizar faces e documenta como evoluir para um reconhecimento facial real com uma base própria.

## 1. instalação e importação das bibliotecas

No Google Colab, o OpenCV geralmente já vem instalado. Mesmo assim, a célula abaixo garante que as bibliotecas usadas no projeto estejam disponíveis.

In [ ]:
!pip install -q opencv-python matplotlib numpy

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

## 2. detecção facial

Detecção facial é a etapa que encontra as regiões da imagem onde existem rostos. Neste projeto, essa parte é feita com o Haar Cascade do OpenCV, um detector pré-treinado simples e bastante usado em exemplos de visão computacional.

## 3. reconhecimento ou classificação facial

Reconhecimento facial é a etapa que tenta identificar quem é a pessoa detectada. Para isso, normalmente seria necessário ter uma base de imagens por pessoa e algum modelo capaz de comparar ou classificar as faces.

Aqui, essa etapa foi representada por labels simulados. Isso mostra o fluxo final esperado, mas sem afirmar que houve treinamento de um modelo próprio.

## 4. carregamento da imagem de exemplo

Faça upload de uma imagem com uma ou mais faces. O notebook vai salvar a imagem como `images/input/input.jpg`.

In [ ]:
input_dir = Path('images/input')
output_dir = Path('images/output')
input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

if not uploaded:
    raise ValueError('nenhuma imagem foi enviada')

uploaded_name = next(iter(uploaded))
input_path = input_dir / 'input.jpg'

with open(input_path, 'wb') as image_file:
    image_file.write(uploaded[uploaded_name])

image = cv2.imread(str(input_path))

if image is None:
    raise ValueError('não foi possível carregar a imagem enviada')

print(f'imagem carregada: {input_path}')

## 5. detecção das faces com OpenCV

In [ ]:
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_detector = cv2.CascadeClassifier(cascade_path)

if face_detector.empty():
    raise RuntimeError('não foi possível carregar o Haar Cascade do OpenCV')

faces = face_detector.detectMultiScale(
    gray_image,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(40, 40),
)

print(f'faces detectadas: {len(faces)}')

## 6. criação de labels simulados

Cada face detectada recebe um nome simulado. Em um sistema real, esse nome viria de um classificador treinado ou de uma comparação com embeddings faciais.

In [ ]:
labels = [f'pessoa_{index}' for index in range(1, len(faces) + 1)]
labels

## 7. desenho das bounding boxes e nomes

In [ ]:
result = image.copy()

for label, (x, y, width, height) in zip(labels, faces):
    cv2.rectangle(
        result,
        (x, y),
        (x + width, y + height),
        (0, 255, 0),
        2,
    )

    label_y = max(y - 10, 20)
    cv2.putText(
        result,
        label,
        (x, label_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )

## 8. salvamento do resultado

In [ ]:
output_path = output_dir / 'face_detection_result.jpg'
cv2.imwrite(str(output_path), result)

print(f'resultado salvo em: {output_path}')

## 9. visualização do resultado

In [ ]:
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(result_rgb)
plt.axis('off')
plt.show()

## 10. como evoluir para reconhecimento real

Para transformar este projeto em reconhecimento facial real, o caminho seria montar uma base própria organizada por pessoa:

```text
dataset/
├── pessoa_1/
├── pessoa_2/
└── pessoa_3/
```

Cada pasta teria várias imagens da mesma pessoa. Depois disso, seria possível detectar e recortar faces, extrair características e treinar um classificador. Outra opção seria usar embeddings faciais gerados por modelos mais modernos e comparar a nova face com as faces conhecidas.

## 11. conclusão

O notebook implementa a etapa de detecção facial com OpenCV e simula a etapa de reconhecimento com labels. O resultado é uma imagem final com as faces marcadas, pronta para ser usada na entrega do desafio e como base para uma evolução futura com dataset próprio.